In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("student.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

student_data = [
(1,"Nguyen Minh Hoang","May Tinh",12,6.7),
(2,"Tran Thi Lan","Kinh Te",34,9.2),
(3,"Pham Van Nam","Toan Tin",None,7.9),
(4,"Le Thanh Huyen","Toan Tin",20,7.2),
(5,"Vu Quoc Anh","May Tinh",24,8.0),
(6,"Dang Thuy Linh","May Tinh",24,5.5),
(7,"Bui Tien Dung","Kinh Te",34,9.2),
(8,"Ho Ngoc Mai","Toan Tin",20,8.8),
(9,"Duong Huu Phuc","Kinh Te",None,7.2),
(10,"Cao Thi Hanh","May Tinh",None,7.0)
]

course_data = [
(12,"Giai tich"),
(34,"Thong ke"),
(26,"Tin hoc")
]

cursor.executemany("INSERT INTO student VALUES (?,?,?,?,?)", student_data)
cursor.executemany("INSERT INTO course VALUES (?,?)", course_data)

conn.commit()

In [4]:
print("Bảng student:")
display(pd.read_sql("SELECT * FROM student", conn))

print("\nBảng course:")
display(pd.read_sql("SELECT * FROM course", conn))

Bảng student:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0



Bảng course:


,id,course_name
0,12,Giai tich
1,34,Thong ke
2,26,Tin hoc


## 1. Kết nối hai bảng trên theo những cách:

In [5]:
# Sử dụng CROSS JOIN để tạo ra kết quả Cartesian
query_cartesian = """
SELECT *
FROM student, course
"""

df_cartesian = pd.read_sql(query_cartesian, conn)
df_cartesian

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


Kết quả cho thấy hai bảng được kết hợp với nhau mà không sử dụng điều kiện liên kết, trong đó mỗi bản ghi của bảng student được ghép với tất cả các bản ghi của bảng course. Do đó, kết quả trả về bao gồm toàn bộ các tổ hợp có thể giữa sinh viên và môn học

In [6]:
# Sử dụng INNER JOIN để kết hợp dữ liệu từ hai bảng dựa trên khóa ngoại
query_inner = """
SELECT *
FROM student s
INNER JOIN course c
ON s.course_id = c.id
"""

df_inner = pd.read_sql(query_inner, conn)
df_inner

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


Kết quả INNER JOIN cho thấy hai bảng được liên kết với nhau thông qua điều kiện course_id của bảng student bằng với id của bảng course, chỉ những sinh viên có mã môn học trùng khớp trong bảng course mới được giữ lại. Kết quả trả về bao gồm thông tin của sinh viên và tên môn học tương ứng mà sinh viên đó tham gia.

In [ ]:
# Sử dụng LEFT JOIN để lấy tất cả dữ liệu từ bảng student và kết hợp với bảng course
query_left = """
SELECT *
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
"""

df_left = pd.read_sql(query_left, conn)
df_left

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,NaN
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,NaN
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,NaN
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,NaN
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,NaN
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,NaN
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,NaN


Kết quả LEFT JOIN cho thấy hai bảng được liên kết thông qua điều kiện course_id của bảng student bằng với id của bảng course. Tất cả các bản ghi từ bảng student được giữ lại, kể cả khi không tìm thấy giá trị tương ứng trong bảng course. Đối với những trường hợp không có sự khớp, các cột từ bảng course sẽ nhận giá trị NULL.

In [8]:
# Sử dụng RIGHT JOIN để lấy tất cả dữ liệu từ bảng course và kết hợp với bảng student
query_right = """
SELECT *
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
"""

df_right = pd.read_sql(query_right, conn)
df_right

,id,course_name,student_id,name,class,course_id,score
0,12,Giai tich,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,34,Thong ke,2.0,Tran Thi Lan,Kinh Te,34.0,9.2
2,34,Thong ke,7.0,Bui Tien Dung,Kinh Te,34.0,9.2
3,26,Tin hoc,NaN,NaN,NaN,NaN,NaN


Kết quả RIGHT JOIN cho thấy hai bảng được liên kết thông qua điều kiện course_id của bảng student bằng với id của bảng course. Tất cả các bản ghi từ bảng course được giữ lại, kể cả khi không có sinh viên tương ứng trong bảng student. Đối với những trường hợp không có sự khớp, các cột từ bảng student sẽ nhận giá trị NULL.

In [10]:
# Sử dụng FULL OUTER JOIN để lấy tất cả dữ liệu từ cả hai bảng kết hợp với nhau
query_full = """
SELECT 
    s.student_id,
    s.name,
    s.class,
    s.course_id,
    s.score,
    c.id,
    c.course_name
FROM student s
LEFT JOIN course c
ON s.course_id = c.id

UNION

SELECT 
    s.student_id,
    s.name,
    s.class,
    s.course_id,
    s.score,
    c.id,
    c.course_name
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
"""

df_full = pd.read_sql(query_full, conn)
df_full

,student_id,name,class,course_id,score,id,course_name
0,NaN,NaN,NaN,NaN,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,NaN
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,NaN
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,NaN
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,NaN
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,NaN
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,NaN


Kết quả FULL OUTER JOIN cho thấy hai bảng được liên kết thông qua điều kiện course_id của bảng student bằng với id của bảng course. Tất cả các bản ghi từ cả hai bảng đều được giữ lại. Những bản ghi không có sự khớp giữa hai bảng sẽ có các cột tương ứng ở bảng còn lại mang giá trị NULL.

## 2. Cập nhật những giá trị course_id còn thiếu trong bảng student, loại bỏ những bản ghi tham gia những môn học không tồn tại bảng course

In [11]:
cursor.execute("""
UPDATE student
SET course_id = 26
WHERE course_id IS NULL
""")

In [12]:
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")

conn.commit()

Sau khi cập nhật các giá trị course_id bị thiếu thành giá trị hợp lệ trong bảng course, câu lệnh DELETE chỉ loại bỏ các bản ghi có course_id không tồn tại (20, 24), cụ thể student_id (4,5,6,8) đã bị xóa, trong khi các bản ghi trước đó bị thiếu đã được giữ lại.

### a. Tổng số sinh viên, điểm trung bình trong mỗi lớp

In [ ]:
query_a = """
SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score),2) AS avg_score
FROM student
GROUP BY class
"""
df_a = pd.read_sql(query_a, conn)
df_a

,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


Kết quả cho thấy số lượng sinh viên và điểm trung bình của từng lớp được tính toán dựa trên các bản ghi hợp lệ sau khi xử lý dữ liệu. Lớp Kinh Tế có số lượng sinh viên nhiều nhất là 3 sinh viên và đồng thời có điểm trung bình cao nhất 8.53, lớp Máy Tính có 2 sinh viên với điểm trung bình thấp hơn 6.85, trong khi lớp Toán Tin có 1 sinh viên với điểm trung bình là 7.9.

### b. Tổng số sinh viên, điểm trung bình của từng môn học

In [ ]:
query_b = """
SELECT c.course_name,
       COUNT(*) AS total_students,
       ROUND(AVG(s.score),2) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
df_b = pd.read_sql(query_b, conn)
df_b

,course_name,total_students,avg_score
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37


Kết quả cho thấy số lượng sinh viên và điểm trung bình của từng môn học đã được tính toán. Môn Thống kê có điểm trung bình cao nhất là 9.2 với 2 sinh viên đóng góp, môn Tin học có số lượng sinh viên nhiều nhất là 3 với điểm trung bình 7.37 và môn Giải tích có 1 sinh viên và điểm trung bình thấp nhất là 6.7.

### c. Phân loại thi đua theo từng môn học

In [ ]:
query_c = """
SELECT c.course_name,
       ROUND(AVG(s.score),2) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
           WHEN AVG(s.score) BETWEEN 6.0 AND 8.9 THEN 'Tot'
           ELSE 'Kem'
       END AS ranking
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
df_c = pd.read_sql(query_c, conn)
df_c

,course_name,avg_score,ranking
0,Giai tich,6.70,Tot
1,Thong ke,9.20,Xuat sac
2,Tin hoc,7.37,Tot


Kết quả cho thấy các môn học đã được phân loại dựa trên điểm trung bình. Môn Thống kê được xếp loại Xuất sắc do có điểm trung bình từ 9.0 trở lên, các môn Giải tích và Tin học được xếp loại Tốt do có điểm trung bình nằm trong khoảng từ 6.0 đến 8.9. Không có môn học nào bị xếp loại Kém.

## 3. Xếp hạng sinh viên
### a. Điểm số 

In [ ]:
query_3a = """
SELECT *,
       RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
"""
df_3a = pd.read_sql(query_3a, conn)
df_3a

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


Sinh viên được xếp hạng theo điểm giảm dần, trong đó hai sinh viên có điểm cao nhất cùng đứng hạng 1 và sinh viên có điểm thấp nhất nên đứng cuối.

In [19]:
# Top 3 sinh viên có điểm số cao nhất
top3_high = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
""", conn)

top3_high

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


In [20]:
# Top 3 sinh viên có điểm số thâp nhất
top3_low = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score ASC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
""", conn)

top3_low

,student_id,name,class,course_id,score,rank_score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3


### b. Điểm số theo lớp học

In [21]:
query_3b = """
SELECT *,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student
"""
df_3b = pd.read_sql(query_3b, conn)
df_3b

,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


Kết quả cho thấy sinh viên được xếp hạng theo điểm số trong từng lớp học. Ở lớp Kinh Tế, hai sinh viên có điểm cao nhất nên cùng xếp hạng 1, trong khi Duong HUu Phuc có điểm thấp hơn nên xếp hạng 3. Ở lớp Máy Tính, Cao Thi Hanh đứng hạng 1 với điểm cao hơn. Ở lớp Toán Tin, Pham Van Nam là sinh viên duy nhất nên đứng hạng 1.

In [22]:
# Top 3 sinh viên có điểm số cao nhất trong mỗi lớp
top3_class = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
""", conn)

top3_class

,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


In [23]:
# Top 3 sinh viên có điểm số thấp nhất trong mỗi lớp
bottom3_class = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
""", conn)

bottom3_class

,student_id,name,class,course_id,score,rank_in_class
0,9,Duong Huu Phuc,Kinh Te,26,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,2
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
4,10,Cao Thi Hanh,May Tinh,26,7.0,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


### c. Điểm số theo mã môn học


In [24]:
query_3c = """
SELECT *,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
FROM student
"""
df_3c = pd.read_sql(query_3c, conn)
df_3c

,student_id,name,class,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,1


Kết quả cho thấy sinh viên được xếp hạng theo điểm số trong từng môn học. Ở môn Giải tích, Nguyen Minh Hoang là sinh viên duy nhất nên đứng hạng 1. Ở môn Tin học, Pham Van Nam có điểm cao nhất nên đứng hạng 1, tiếp theo là Duong Huu Phuc và Cao Thi Hanh. Ở môn Thống kê, hai sinh viên Tran Thi Lan và Bui Tien Dung có cùng điểm cao nhất nên cùng xếp hạng 1.

In [25]:
# Top 3 sinh viên có điểm số cao nhất trong mỗi môn học
top3_course = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
    FROM student
)
WHERE rank_in_course <= 3
""", conn)

top3_course

,student_id,name,class,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,1


In [26]:
# Top 3 sinh viên có điểm số thấp nhất trong mỗi môn học
bottom3_course = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS rank_in_course
    FROM student
)
WHERE rank_in_course <= 3
""", conn)

bottom3_course

,student_id,name,class,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,3,Pham Van Nam,Toan Tin,26,7.9,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,1


## 4. Bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng student để xác định thời gian tốt nghiệp của từng bạn

In [27]:
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date DATETIME
""")
conn.commit()

In [28]:
cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' || (
    SELECT rank_score
    FROM (
        SELECT student_id,
               RANK() OVER (ORDER BY score DESC) AS rank_score
        FROM student
    ) t
    WHERE t.student_id = student.student_id
) || ' days')
""")

conn.commit()

In [29]:
df = pd.read_sql("SELECT * FROM student", conn)
df

,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-07 10:04:29
1,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-02 10:04:29
2,3,Pham Van Nam,Toan Tin,26,7.9,2026-04-04 10:04:29
3,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-02 10:04:29
4,9,Duong Huu Phuc,Kinh Te,26,7.2,2026-04-05 10:04:29
5,10,Cao Thi Hanh,May Tinh,26,7.0,2026-04-06 10:04:29


Kết quả cho thấy thời gian tốt nghiệp của sinh viên được xác định dựa trên thứ hạng theo điểm số. Thứ hạng được tính bằng hàm RANK() theo thứ tự điểm giảm dần, sau đó được sử dụng để cộng thêm số ngày tương ứng vào thời gian hiện tại. Kết quả cho thấy sinh viên có điểm cao hơn sẽ có thời gian tốt nghiệp sớm hơn, trong khi sinh viên có điểm thấp hơn sẽ tốt nghiệp muộn hơn. Các trường hợp có cùng điểm sẽ có cùng thứ hạng và cùng thời gian tốt nghiệp.